# Applying `utils.py` to the MIT-BIH Arrhythmia Database

Every MIT-BIH record has exactly 2 leads, but *which* 2 varies by record --
most are `MLII` + `V1`, some (like record 100) are `MLII` + `V5`, a few use
other combinations. `utils.py` handles this automatically: `MLII` always maps
to the standard `II` slot, the second lead is auto-detected per record, and
the remaining 10 lead slots (out of the generic 12-lead layout `utils.py`
supports) are filled with `NaN` and simply ignored. This lets the exact same
code be reused later on a true 12-lead dataset without any changes.

The whole pipeline (load signal -> R-peaks -> heartbeats -> embeddings ->
save to `.npz`) lives entirely in `utils.main()`. This notebook:

1. Runs it on a single record as a smoke test.
2. Runs it on **all 48 records** of the MIT-BIH Arrhythmia Database.
3. Exports each record's results as `.json` (via `export_atlas_json`) **into
   Google Drive**, ready to open with `ECG_Software_v3.html`.


In [ ]:
!pip install wfdb -q
!pip install neurokit2 -q
!pip install umap-learn -q


In [ ]:
import numpy as np
import os
from utils import main, load_results, export_atlas_json, AAMI_CLASS_NAMES


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Paths

`RECORD_PATH_BASE` points at the folder containing the MIT-BIH `.dat`/`.hea`/`.atr`
files (record path is built internally as `f"{RECORD_PATH_BASE}{record_number}"`).

`NPZ_OUTPUT_DIR` and `JSON_OUTPUT_DIR` are where `main()`'s `.npz` results and
the atlas-viewer `.json` exports get saved, respectively -- both **in Google
Drive**, in a new `atlas_exports/` folder next to the dataset, so everything
persists across Colab sessions.


In [ ]:
RECORD_PATH_BASE = "/content/drive/My Drive/Imaging Physics Group/Heart Arrhythmia Detection/mit-bih-arrhythmia-database-1.0.0/"

EXPORT_BASE_DIR = "/content/drive/My Drive/Imaging Physics Group/Heart Arrhythmia Detection/atlas_exports/"
NPZ_OUTPUT_DIR  = os.path.join(EXPORT_BASE_DIR, "npz")
JSON_OUTPUT_DIR = os.path.join(EXPORT_BASE_DIR, "json")

os.makedirs(NPZ_OUTPUT_DIR, exist_ok=True)
os.makedirs(JSON_OUTPUT_DIR, exist_ok=True)

print("Records read from :", RECORD_PATH_BASE)
print("NPZ saved to       :", NPZ_OUTPUT_DIR)
print("JSON saved to      :", JSON_OUTPUT_DIR)


## Run it (single record smoke test)

Point at one record first to confirm paths/leads are correct before running
the full database below. `leads_for_embeddings` is left at its default
(`None`), so `main()` auto-detects whichever 2 leads this record actually has.


In [ ]:
RECORD_NUMBER = 100
OUTPUT_DIR = NPZ_OUTPUT_DIR

# sanity check that the record files exist before running
for ext in ["dat", "hea", "atr"]:
    fpath = os.path.join(RECORD_PATH_BASE, f"{RECORD_NUMBER}.{ext}")
    print(fpath, "->", "FOUND" if os.path.exists(fpath) else "MISSING")

results = main(RECORD_NUMBER, RECORD_PATH_BASE, output_dir=OUTPUT_DIR)


In [ ]:
print(results["ecg_signals"])
print(results["heartbeats"])
print("Leads embedded:", list(results["embeddings"].keys()))
print("Saved to:", results["output_path"])


## Reloading a saved result

`load_results` restores everything `main()` saved, including embeddings as
one `(12, N, 2)` array per method (`umap`/`tsne`/`pca`) -- same `(12, ...)`
layout as `signal`/`heartbeats`, so you index it the same way:
`embeddings["umap"]["II"]` -> shape `(N, 2)`.


In [ ]:
reloaded = load_results(results["output_path"])

print(reloaded["ecg_signals"])
print(reloaded["heartbeats"])
print(reloaded["embeddings"]["umap"])  # LeadArray, shape (12, N, 2)
print("UMAP embedding for lead II:", reloaded["embeddings"]["umap"]["II"].shape)

if reloaded["aami_labels"] is not None:
    print("AAMI label counts:",
          dict(zip(*np.unique(reloaded["aami_labels"], return_counts=True))))


## Export the smoke-test record for the ECG Atlas Visualization HTML

`export_atlas_json` converts a `main()` results dict into the JSON schema the
`ECG_Software_v3.html` viewer expects (`signal_data`, `heartbeats_data`,
`r_peaks`, `aami_labels`, `embeddings`, one entry per lead -- both leads, since
`main()` now embeds every lead the record has). Saved straight into
`JSON_OUTPUT_DIR` on Drive.


In [ ]:
json_path = os.path.join(JSON_OUTPUT_DIR, f"{RECORD_NUMBER}.json")
export_atlas_json(results, json_path)


## Run the full MIT-BIH database and save every JSON export to Drive

All 48 records of the MIT-BIH Arrhythmia Database (record numbers per the
official PhysioNet record list). For each record this:

1. Runs the full pipeline via `main()` (leads auto-detected per record),
   saving the raw `.npz` pipeline output to `NPZ_OUTPUT_DIR` on Drive.
2. Exports the atlas-viewer `.json` to `JSON_OUTPUT_DIR` on Drive.

A record that fails (e.g. a corrupt/missing file) is logged and skipped
rather than stopping the whole batch -- check the summary printed at the end.
This can take a while for all 48 records; `plot=False, verbose=False` keeps
the output from each individual call from flooding the notebook.


In [ ]:
from tqdm.auto import tqdm

ALL_MITDB_RECORDS = [
    100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
    111, 112, 113, 114, 115, 116, 117, 118, 119,
    121, 122, 123, 124,
    200, 201, 202, 203, 205,
    207, 208, 209, 210,
    212, 213, 214, 215,
    217,
    219, 220, 221, 222, 223,
    228,
    230, 231, 232, 233, 234,
]
assert len(ALL_MITDB_RECORDS) == 48

batch_results = {}
failed = {}

for rec in tqdm(ALL_MITDB_RECORDS, desc="MIT-BIH records"):
    try:
        res = main(
            rec, RECORD_PATH_BASE,
            output_dir=NPZ_OUTPUT_DIR,
            leads_for_embeddings=None,   # auto-detect this record's 2 leads
            plot=False, verbose=False,
        )
        json_path = os.path.join(JSON_OUTPUT_DIR, f"{rec}.json")
        export_atlas_json(res, json_path, verbose=False)

        batch_results[rec] = res
        print(f"Record {rec}: OK -> leads {list(res['embeddings'].keys())}, "
              f"{res['heartbeats'].shape[1]} beats -> {json_path}")
    except Exception as e:
        failed[rec] = str(e)
        print(f"Record {rec}: FAILED -> {e}")

print()
print(f"Done: {len(batch_results)}/{len(ALL_MITDB_RECORDS)} records succeeded.")
if failed:
    print("Failed records:")
    for rec, err in failed.items():
        print(f"  {rec}: {err}")


## Summary

Quick check of what landed in Drive, and the lead pairing `main()`
auto-detected per record (useful for confirming records like 100 correctly
picked up `V5` instead of `V1`).


In [ ]:
json_files = sorted(f for f in os.listdir(JSON_OUTPUT_DIR) if f.endswith(".json"))
print(f"{len(json_files)} JSON files in {JSON_OUTPUT_DIR}:")
print(json_files)

print()
print("Leads used per record:")
for rec, res in batch_results.items():
    print(f"  {rec}: {list(res['embeddings'].keys())}")
